In [ ]:
import torch
import torch.nn as nn

T = 1000  # total diffusion timesteps

# ---------- Forward process: q(x_t | x_{t-1}) ----------

def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, noise=None):
    """Sample x_t directly from x0 using the closed-form of q(x_t | x0):
       x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise"""
    if noise is None:
        noise = torch.randn_like(x0)
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
    return x_t, noise

# ---------- Reverse process: p_theta(x_{t-1} | x_t) ----------

class TinyDenoiser(nn.Module):
    """A small CNN that predicts the noise added at timestep t (epsilon-theta)."""
    def __init__(self, channels=3):
        super().__init__()
        self.time_embed = nn.Embedding(T, 32)
        self.net = nn.Sequential(
            nn.Conv2d(channels, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, channels, 3, padding=1),
        )

    def forward(self, x, t):
        # (a full model injects the time embedding into every conv block;
        #  simplified here to keep the example short)
        return self.net(x)

@torch.no_grad()
def reverse_denoising_loop(model, shape=(1, 3, 64, 64)):
    """Iteratively samples x_{t-1} from x_t for t = T-1 ... 0, generating
    a 64x64 image from pure Gaussian noise."""
    x = torch.randn(shape)

    for t in reversed(range(T)):
        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]

        predicted_noise = model(x, torch.tensor([t]))

        mean = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise
        )

        if t > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = mean + sigma_t * noise
        else:
            x = mean   # final step: no noise added

    return x  # final generated 64x64 image tensor


def train_step(model, optimizer, x0_batch):
    """One DDPM training step: predict the noise added at a random timestep."""
    b = x0_batch.shape[0]
    t = torch.randint(0, T, (b,))
    x_t, true_noise = forward_diffusion(x0_batch, t)

    predicted_noise = model(x_t, t)
    loss = nn.functional.mse_loss(predicted_noise, true_noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


if __name__ == "__main__":
    torch.manual_seed(0)
    model = TinyDenoiser()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    fake_images = torch.randn(4, 3, 64, 64)  # stand-in for a real image batch
    for step in range(3):
        loss = train_step(model, optimizer, fake_images)
        print(f"train step {step}: loss={loss:.4f}")

    print("Running reverse denoising loop (this simulates all", T, "steps)...")
    generated = reverse_denoising_loop(model)
    print("Generated image tensor shape:", generated.shape)

---
## Task 10: DDPM Forward & Reverse Latent Optimization